In [ ]:
!nvidia-smi

Mon Apr 13 15:45:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile vector_add.cu
#include <cuda_runtime.h>
#include <bits/stdc++.h>
using namespace std;
using namespace std::chrono;

__global__ void add(int *A, int *B, int *C, int size) {
    int id = blockDim.x * blockIdx.x + threadIdx.x;
    if (id < size) {
        C[id] = A[id] + B[id];
    }
}

void initialize(int *vector, int size) {
    for (int i = 0; i < size; i++) vector[i] = rand() % 10;
}

void print(int *vector, int size) {
    for (int i = 0; i < size; i++) cout << vector[i] << " ";
    cout << endl;
}

int main() {
    int vectorSize = 8;
    size_t vectorBytes = vectorSize * sizeof(int);

    int *A = new int[vectorSize];
    int *B = new int[vectorSize];
    int *C = new int[vectorSize];

    initialize(A, vectorSize);
    initialize(B, vectorSize);

    cout << "Vector A: "; print(A, vectorSize);
    cout << "Vector B: "; print(B, vectorSize);

    int *X, *Y, *Z;
    cudaMalloc(&X, vectorBytes);
    cudaMalloc(&Y, vectorBytes);
    cudaMalloc(&Z, vectorBytes);

    cudaMemcpy(X, A, vectorBytes, cudaMemcpyHostToDevice);
    cudaMemcpy(Y, B, vectorBytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (vectorSize + threadsPerBlock - 1) / threadsPerBlock;

    auto start = high_resolution_clock::now();
    add<<<blocksPerGrid, threadsPerBlock>>>(X, Y, Z, vectorSize);
    cudaDeviceSynchronize();
    auto stop = high_resolution_clock::now();

    cudaMemcpy(C, Z, vectorBytes, cudaMemcpyDeviceToHost);

    auto duration = duration_cast<microseconds>(stop - start);
    cout << "Result (A+B): "; print(C, vectorSize);
    cout << "Time: " << duration.count() << " microseconds\n";

    delete[] A; delete[] B; delete[] C;
    cudaFree(X); cudaFree(Y); cudaFree(Z);
    return 0;
}

Writing vector_add.cu


In [ ]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./vector_add

Vector A: 3 6 7 5 3 5 6 2 
Vector B: 9 1 2 7 0 9 3 6 
Result (A+B): 12 7 9 12 3 14 9 8 
Time: 94377 microseconds


In [ ]:
%%writefile matrix_mul.cu
#include <cuda_runtime.h>
#include <bits/stdc++.h>
using namespace std;
using namespace std::chrono;

__global__ void multiply(int *A, int *B, int *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        int sum = 0;
        for (int k = 0; k < N; k++)
            sum += A[row * N + k] * B[k * N + col];
        C[row * N + col] = sum;
    }
}

void initialize(int *matrix, int N) {
    for (int i = 0; i < N * N; i++) matrix[i] = rand() % 10;
}

void print(int *matrix, int N) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) cout << matrix[i * N + j] << " ";
        cout << endl;
    }
    cout << endl;
}

int main() {
    int N = 4;
    size_t bytes = N * N * sizeof(int);

    int *A = new int[N * N];
    int *B = new int[N * N];
    int *C = new int[N * N];

    initialize(A, N);
    initialize(B, N);

    cout << "Matrix A:\n"; print(A, N);
    cout << "Matrix B:\n"; print(B, N);

    int *X, *Y, *Z;
    cudaMalloc(&X, bytes);
    cudaMalloc(&Y, bytes);
    cudaMalloc(&Z, bytes);

    cudaMemcpy(X, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(Y, B, bytes, cudaMemcpyHostToDevice);

    int THREADS = 16;
    dim3 threads(THREADS, THREADS);
    dim3 blocks((N + THREADS - 1) / THREADS, (N + THREADS - 1) / THREADS);

    auto start = high_resolution_clock::now();
    multiply<<<blocks, threads>>>(X, Y, Z, N);
    cudaDeviceSynchronize();
    auto stop = high_resolution_clock::now();

    cudaMemcpy(C, Z, bytes, cudaMemcpyDeviceToHost);

    auto duration = duration_cast<microseconds>(stop - start);
    cout << "Result (A x B):\n"; print(C, N);
    cout << "Time: " << duration.count() << " microseconds\n";

    delete[] A; delete[] B; delete[] C;
    cudaFree(X); cudaFree(Y); cudaFree(Z);
    return 0;
}

Writing matrix_mul.cu


In [ ]:
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./matrix_mul

Matrix A:
3 6 7 5 
3 5 6 2 
9 1 2 7 
0 9 3 6 

Matrix B:
0 6 2 6 
1 8 7 9 
2 0 2 3 
7 5 9 2 

Result (A x B):
55 91 107 103 
31 68 71 85 
54 97 92 83 
57 102 123 102 

Time: 30035 microseconds
